# arXiv 논문 검색/수집/삭제 클라이언트 (Google Colab용)

**용도**: 팀 공유 서버(EC2 FastAPI)에 접속해 논문 **검색·조회·수집·삭제**를 하는 팀원용 노트북. 저장소 클론 없이 **이 노트북 하나만으로** 코랩에서 바로 실행됩니다.

**전체 프로세스에서의 역할**: `client_example.py` + `collect_and_push.py`의 코랩 버전 — 관심사 프로필로 라벨링 후보를 추출하고, 필요 시 특정 기간을 분산 수집하거나 잘못 들어간 데이터를 삭제합니다.

**실행 방법**:
1. 이 파일을 [Google Colab](https://colab.research.google.com)에서 열기 (GitHub에서 열기 또는 업로드)
2. 1~2번 셀(접속 설정, 함수 정의)을 먼저 실행
3. 필요한 기능의 섹션으로 이동해 실행

**사전 조건**: EC2 서버(`uvicorn app:app`)가 켜져 있어야 하고, 팀에서 공유받은 서버 주소와 API 키가 필요합니다. 삭제 기능(8번)은 서버가 최신 `app.py`(`/papers/delete` 엔드포인트 포함)로 배포되어 있어야 동작합니다.

> ⚠️ **API 키를 노트북에 직접 타이핑하지 마세요.** 아래 셀이 실행 시 입력창(`getpass`)으로 받기 때문에 키가 노트북 파일에 저장되지 않습니다. 이 노트북은 GitHub에 커밋되는 파일입니다.

## 1. 접속 정보 설정

- `API_BASE`: 서버 주소 (예: `http://<EC2-IP>:8000`) — 비밀값은 아니지만 팀 외부 공유 금지
- `API_KEY`: 실행하면 나오는 입력창에 붙여넣기 (화면에 표시되지 않음)

코랩 **보안 비밀**(왼쪽 🔑 아이콘)에 `ARXIV_API_BASE`, `ARXIV_API_KEY`를 등록해두면 입력 없이 자동으로 불러옵니다.

In [ ]:
from getpass import getpass

API_BASE = None
API_KEY = None

# 1순위: 코랩 보안 비밀(Secrets)에서 읽기
try:
    from google.colab import userdata
    try:
        API_BASE = userdata.get("ARXIV_API_BASE")
        API_KEY = userdata.get("ARXIV_API_KEY")
        print("코랩 보안 비밀에서 접속 정보를 불러왔습니다.")
    except Exception:
        pass
except ImportError:
    pass  # 코랩이 아닌 환경 (로컬 주피터 등)

# 2순위: 직접 입력
if not API_BASE:
    API_BASE = input("API_BASE (예: http://1.2.3.4:8000): ").strip()
if not API_KEY:
    API_KEY = getpass("API_KEY (입력해도 화면에 안 보임): ").strip()

API_BASE = API_BASE.rstrip("/")
HEADERS = {"X-API-Key": API_KEY}
print(f"서버: {API_BASE}")

## 2. 클라이언트 함수 정의

`client_example.py`와 동일한 함수들입니다 (requests만 사용, 무거운 라이브러리 불필요).

In [ ]:
import requests
import pandas as pd


def health():
    """서버 생존 확인 + 전체 논문 수 (인증 불필요)."""
    resp = requests.get(f"{API_BASE}/health", timeout=10)
    resp.raise_for_status()
    return resp.json()


def search(query: str, top_k: int = 20, category: str = None):
    """관심사 프로필 텍스트로 유사 논문 검색.

    Args:
        query: 검색 텍스트 (영어 권장 — 임베딩 모델이 영어 모델)
        top_k: 반환 논문 수
        category: 주 카테고리 필터 (예: "cs.RO"), 생략 시 전체

    Returns:
        논문 dict 리스트. score는 cosine distance — **작을수록 유사**.
    """
    payload = {"query": query, "top_k": top_k}
    if category:
        payload["category"] = category
    resp = requests.post(f"{API_BASE}/search", json=payload, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return resp.json()


def get_paper(arxiv_id: str):
    """단일 논문 상세 조회 (초록 원문 포함)."""
    resp = requests.get(f"{API_BASE}/papers/{arxiv_id}", headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return resp.json()


def get_papers(arxiv_ids: list):
    """여러 논문 상세 조회 (라벨링 후보 리스트 확인용)."""
    resp = requests.get(
        f"{API_BASE}/papers",
        params={"ids": ",".join(arxiv_ids)},
        headers=HEADERS,
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()


def search_df(query: str, top_k: int = 20, category: str = None) -> pd.DataFrame:
    """search() 결과를 보기 좋은 DataFrame으로 반환."""
    results = search(query, top_k=top_k, category=category)
    df = pd.DataFrame(results)
    if df.empty:
        return df
    cols = ["arxiv_id", "score", "title", "primary_category", "submitted_date", "abs_url"]
    return df[[c for c in cols if c in df.columns]]


print("함수 정의 완료")

## 3. 접속 확인

In [ ]:
health()
# {'status': 'ok', 'paper_count': ...} 형태가 나오면 정상

## 4. 논문 검색

- **쿼리는 영어로** 작성하세요 (임베딩 모델 `bge-small-en`이 영어 모델)
- `score`는 cosine **거리**라서 **작을수록 유사**합니다 (정렬 방향 주의)

In [ ]:
# 간단한 주제 검색
search_df("speculative decoding for efficient LLM inference", top_k=10, category="cs.LG")

## 5. 관심사 프로필로 라벨링 후보 추출

노션 프로필 페이지의 프로필 본문(영어)을 그대로 쿼리로 사용하고, 라벨링용으로는 top 30~50 정도를 넉넉히 추출합니다.

In [ ]:
# 예: P1. 언어 조건 로봇 매니퓰레이션 프로필
profile_p1 = (
    "I am interested in language-conditioned robot manipulation, "
    "especially how vision-language-action models interpret ambiguous or "
    "underspecified natural language instructions and ground them into "
    "executable actions."
)

candidates = search_df(profile_p1, top_k=30, category="cs.RO")
candidates

In [ ]:
# 후보 목록을 CSV로 저장 (라벨링 시트 만들 때 사용)
candidates.to_csv("labeling_candidates_p1.csv", index=False)
print("labeling_candidates_p1.csv 저장 완료 —", len(candidates), "건")

## 6. 논문 상세 조회

In [ ]:
# 위 검색 결과 중 첫 번째 논문의 상세 정보
if len(candidates) > 0:
    detail = get_paper(candidates.iloc[0]["arxiv_id"])
    print("제목:", detail["title"])
    print("카테고리:", detail["categories"])
    print("제출일:", detail["submitted_date"])
    print("링크:", detail["abs_url"])
    print()
    print(detail["abstract_clean"][:800])

## 7. 데이터 수집 (arXiv → 서버 push)

`collect_and_push.py`와 같은 동작을 코랩에서 수행합니다.
지정한 기간의 논문을 arXiv API에서 직접 수집해 서버 `/papers/ingest`로 전송하면, **서버가 전처리·저장·임베딩까지 처리**합니다.

- arXiv 예의상 요청 간 3초 대기가 들어가므로 **긴 기간은 오래 걸립니다** (수 주 분량 = 수십 분~수 시간)
- 결과가 너무 많으면(창당 약 1만 건↑) arXiv가 500 에러를 내므로 7일 단위 창으로 쪼개 수집합니다. 에러가 잦으면 `chunk_days`를 3으로 줄이세요
- 429/503(rate limit/일시 장애) 시 자동 백오프 재시도, 끝내 실패한 창은 건너뛰고 알려줍니다

In [ ]:
%pip install -q arxiv

In [ ]:
import re
import time
from datetime import datetime, timedelta, timezone

import arxiv

_VERSION_RE = re.compile(r"^(?P<base>.+)v(?P<ver>\d+)$")


def _fetch_window(start, end, categories):
    """한 시간 창의 논문을 arXiv에서 조회해 서버 전송용 dict 리스트로 반환."""
    cat_q = "(" + " OR ".join(f"cat:{c}" for c in categories) + ")"
    fmt = "%Y%m%d%H%M%S"
    query = f"{cat_q} AND submittedDate:[{start.strftime(fmt)} TO {end.strftime(fmt)}]"

    client = arxiv.Client(page_size=100, delay_seconds=3.0, num_retries=3)
    s = arxiv.Search(query=query, sort_by=arxiv.SortCriterion.SubmittedDate, max_results=None)

    out = []
    for r in client.results(s):
        short_id = r.get_short_id()
        m = _VERSION_RE.match(short_id)
        base_id, ver = (m.group("base"), int(m.group("ver"))) if m else (short_id, 1)
        out.append({
            "arxiv_id": base_id,
            "version": ver,
            "title": r.title.strip(),
            "authors": [a.name for a in r.authors],
            "abstract_raw": r.summary.strip(),
            "categories": list(r.categories),
            "primary_category": r.primary_category,
            "comments": r.comment,
            "submitted_date": r.published.isoformat(),
            "updated_date": r.updated.isoformat(),
            "abs_url": r.entry_id,
            "pdf_url": r.pdf_url or "",
        })
    return out


def collect_and_push(start_date: str, end_date: str,
                     categories=("cs.LG", "cs.CV", "cs.RO"),
                     chunk_days: int = 7, batch_size: int = 200,
                     max_retries: int = 5):
    """start_date ~ end_date(포함, YYYY-MM-DD)의 논문을 수집해 서버로 전송.

    서버가 초록 클린업 + 버전 비교 upsert + 임베딩까지 처리한다.
    이미 서버에 있는 논문은 중복 저장되지 않으므로 기간이 겹쳐도 안전.
    """
    start = datetime.strptime(start_date, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    end = datetime.strptime(end_date, "%Y-%m-%d").replace(tzinfo=timezone.utc) + timedelta(days=1)

    total_changed = total_embedded = 0
    window = start
    while window < end:
        w_end = min(end, window + timedelta(days=chunk_days))
        label = f"{window.date()}~{w_end.date()}"

        papers = []
        for attempt in range(1, max_retries + 1):
            try:
                papers = _fetch_window(window, w_end, categories)
                break
            except Exception as e:
                if attempt >= max_retries:
                    print(f"[경고] {label} 조회 {max_retries}회 모두 실패({e}) — 이 구간은 건너뜀. "
                          f"나중에 collect_and_push('{window.date()}', '{w_end.date()}')로 재수집하세요.")
                    papers = []
                    break
                wait = 30 * attempt
                print(f"[경고] {label} 조회 실패({e}), {wait}초 후 재시도 ({attempt}/{max_retries})")
                time.sleep(wait)

        # 중복 제거: 같은 arxiv_id는 최고 버전만, 카테고리는 합집합
        best = {}
        for p in papers:
            k = p["arxiv_id"]
            if k in best:
                best[k]["categories"] = sorted(set(best[k]["categories"]) | set(p["categories"]))
                if p["version"] > best[k]["version"]:
                    cats = best[k]["categories"]
                    best[k] = p
                    best[k]["categories"] = cats
            else:
                best[k] = p
        deduped = list(best.values())
        print(f"[{label}] 조회 {len(papers)}건 -> 중복 제거 후 {len(deduped)}건, 전송 시작")

        for i in range(0, len(deduped), batch_size):
            chunk = deduped[i : i + batch_size]
            resp = requests.post(
                f"{API_BASE}/papers/ingest",
                json={"papers": chunk, "auto_embed": True},
                headers=HEADERS, timeout=300,
            )
            resp.raise_for_status()
            r = resp.json()
            total_changed += r["ingested_or_updated"]
            total_embedded += r["newly_embedded"]
            print(f"  배치 {i // batch_size + 1}: 신규/갱신 {r['ingested_or_updated']}건, 임베딩 {r['newly_embedded']}건")

        window = w_end

    print(f"완료: 총 신규/갱신 {total_changed}건, 임베딩 {total_embedded}건")


print("수집 함수 정의 완료")

In [ ]:
# 사용 예: 특정 기간 수집 (카테고리 기본값: cs.LG, cs.CV, cs.RO)
# collect_and_push("2026-07-01", "2026-07-15")

# 카테고리·창 크기 지정 (cs.LG는 논문량이 많아 chunk_days=3 권장)
# collect_and_push("2026-04-01", "2026-04-30", categories=["cs.LG"], chunk_days=3)

## 8. 데이터 삭제

⚠️ **삭제는 되돌릴 수 없습니다** — 팀 공유 DB에서 지워지므로 **반드시 팀 합의 후** 실행하세요.

- SQLite와 Chroma 벡터에서 함께 삭제됩니다
- 전체 삭제는 위험해서 API로 제공하지 않습니다 (서버에서 `reset_data.py --all` 사용)
- 서버가 최신 `app.py`(`/papers/delete` 포함)로 배포되어 있어야 합니다 (아니면 404)

In [ ]:
def delete_by_ids(arxiv_ids: list):
    """지정한 arxiv_id 논문들을 서버 DB에서 삭제 (실행 전 확인 프롬프트)."""
    print(f"삭제 대상: {len(arxiv_ids)}건 — 예: {arxiv_ids[:5]}")
    if input("삭제하려면 'delete' 입력: ").strip() != "delete":
        print("취소했습니다.")
        return None
    resp = requests.post(
        f"{API_BASE}/papers/delete",
        json={"arxiv_ids": list(arxiv_ids), "confirm": True},
        headers=HEADERS, timeout=300,
    )
    resp.raise_for_status()
    r = resp.json()
    print(f"삭제 {r['deleted']}건, 서버 잔여 {r['remaining']}건")
    return r


def delete_by_category(category: str):
    """primary_category가 일치하는 논문을 일괄 삭제 (카테고리 개편용).
    예: delete_by_category("cs.CL")"""
    print(f"삭제 대상: primary_category = {category} 전체")
    if input(f"'{category}' 논문을 전부 삭제하려면 'delete' 입력: ").strip() != "delete":
        print("취소했습니다.")
        return None
    resp = requests.post(
        f"{API_BASE}/papers/delete",
        json={"category": category, "confirm": True},
        headers=HEADERS, timeout=600,
    )
    resp.raise_for_status()
    r = resp.json()
    print(f"삭제 {r['deleted']}건, 서버 잔여 {r['remaining']}건")
    return r


print("삭제 함수 정의 완료")

In [ ]:
# 사용 예 (주석 해제 후 실행 — 되돌릴 수 없으니 팀 합의 필수)

# 특정 논문 몇 건만 삭제
# delete_by_ids(["2401.12345", "2402.00001"])

# 카테고리 개편으로 빠진 cs.CL 논문 일괄 삭제
# delete_by_category("cs.CL")

## 자주 겪는 문제

| 증상 | 원인 | 해결 |
|---|---|---|
| `ConnectionError` / 타임아웃 | 서버가 꺼져 있거나 주소 오타 | EC2에서 uvicorn 구동 상태, `API_BASE` 확인 |
| `401 Unauthorized` | API 키 불일치 | 팀 공유 키 재확인 후 1번 셀 다시 실행 |
| `404` (papers/delete) | 서버가 구버전 app.py | EC2에서 `git pull` 후 uvicorn 재시작 |
| `400` (papers/delete) | confirm 누락 또는 ids/category 동시 지정 | 함수(delete_by_ids/category)를 통해 호출 |
| 수집 중 HTTP 500 | 창당 결과 1만 건 초과 | `chunk_days`를 3 이하로 축소 |
| 검색 결과가 이상함 | 한국어 쿼리 | 영어로 쿼리 작성 |
| 상위 결과가 안 비슷함 | score를 유사도로 착각 | score는 거리 — 작을수록 유사 |